# 01. Import And Visualize Robots In Pinocchio

This notebook is a compact introduction to the workflow you will use repeatedly later in the course: load a robot, inspect its Pinocchio models, display it, and move it through a few configurations.

## What you should remember
- `load_robot_description(...)` loads a robot together with its `model`, `collision_model`, and `visual_model`.
- `pin.neutral(model)` gives a convenient reference configuration.
- `pin.integrate(model, q, v_dt)` is the safe way to update configurations in Pinocchio.
- Frames and simple primitives are often enough to make the geometry of a model much easier to read.
- `pin.appendModel(...)` lets you build larger systems by attaching one robot model to another.

In [1]:
# %%capture
# !pip install pin viser robot_descriptions numpy scipy matplotlib trimesh

In [1]:
import site
site.main()

In [2]:
%%capture
# !wget -O viz.py https://raw.githubusercontent.com/Atarilab/colab_utils/refs/heads/main/viz.py
import viz

If the next cell gives an error restart the session to load the libraries (ctrl + m + .) or click runtime -> restart session. Then rerun the second and third codeblocks (do not rerun the first block!).

In [3]:
from functools import partial

import time

import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
import trimesh
from robot_descriptions.loaders.pinocchio import load_robot_description

from viz import PinNotebookViz, add_mesh_handle, create_server, update_object_pose

server, share_url = create_server()
notebook_viz = PinNotebookViz(server)
add_mesh_handle = partial(add_mesh_handle, server)

print(f"Open the visualizer here: {share_url}")


╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://mic-powered.share.viser.studio

Open the visualizer here: https://mic-powered.share.viser.studio


(viser) Connection opened (0, 1 total), 6 persistent messages

(viser) Connection closed (0, 0 total)

(viser) Connection opened (1, 1 total), 6 persistent messages

## Load one robot and inspect what Pinocchio gives us
We start with a fixed-base manipulator so the basic workflow is easy to read.

In [4]:
robot = load_robot_description("ur5_description")
model = robot.model
collision_model = robot.collision_model
visual_model = robot.visual_model

print(f"nq = {model.nq}, nv = {model.nv}, njoints = {model.njoints}, nframes = {model.nframes}")
print(f"collision geometries = {len(collision_model.geometryObjects)}")
print(f"visual geometries    = {len(visual_model.geometryObjects)}")
print()
print("First few joint names:")
for name in model.names[:8]:
    print(" ", name)

/home/juan-amaya/miniconda3/envs/contact_rich_robotics/lib/python3.11/importlib/__init__.py:126: FutureWarning: robot_descriptions.ur5_description is deprecated and will switch to the official UR5 model in robot_descriptions.py v2. Use robot_descriptions.ur5_official_description now to migrate early.
  return _bootstrap._gcd_import(name[level:], package, level)


nq = 6, nv = 6, njoints = 7, nframes = 22
collision geometries = 8
visual geometries    = 7

First few joint names:
  universe
  shoulder_pan_joint
  shoulder_lift_joint
  elbow_joint
  wrist_1_joint
  wrist_2_joint
  wrist_3_joint


(viser) Connection closed (1, 0 total)

(viser) Connection opened (2, 1 total), 6 persistent messages

In [ ]:
notebook_viz.attach_robot(robot)
q_neutral = pin.neutral(model)
notebook_viz.display(q_neutral)
q_neutral

## Visualize a few useful frames
Frames are often the quickest way to understand where Pinocchio thinks a base, tool, or sensor lives.

In [ ]:
data = model.createData()
pin.framesForwardKinematics(model, data, q_neutral)
base_frame_id = model.getFrameId("base_link")
tool_frame_id = model.getFrameId("tool0")

notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("base_link", data.oMf[base_frame_id])
notebook_viz.show_frame("tool0", data.oMf[tool_frame_id])

## Visualize simple primitives like boxes and spheres
These primitives are useful when you want to annotate a scene or attach a simple object to a robot frame without building a full visual model.

In [ ]:
notebook_viz.attach_robot(robot)
notebook_viz.display(q_neutral)
pin.framesForwardKinematics(model, data, q_neutral)
tool_frame = data.oMf[tool_frame_id]

world_box = trimesh.creation.box(extents=(0.18, 0.12, 0.12))
world_box.visual.face_colors = [255, 150, 80, 180]

tool_sphere = trimesh.creation.icosphere(radius=0.06, subdivisions=2)
tool_sphere.visual.face_colors = [80, 170, 255, 180]

world_box_handle = add_mesh_handle('/world_box', world_box, pin.SE3(np.eye(3), np.array([0.55, -0.20, 0.10])))
tool_sphere_handle = add_mesh_handle('/tool_sphere', tool_sphere, tool_frame * pin.SE3(np.eye(3), np.array([0.10, 0.0, 0.0])))

notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("tool0", tool_frame)

## Move a primitive together with the tool frame
The next cell makes the robot move and updates the small sphere so that it stays attached to the tool frame.

In [ ]:
q = pin.neutral(model)
dt = 0.02
steps = 180
base_velocity = np.linspace(0.3, 0.9, model.nv)

for k in range(steps):
    v = 0.6 * np.sin(0.05 * k) * base_velocity
    q = pin.integrate(model, q, v * dt)
    notebook_viz.display(q)
    pin.framesForwardKinematics(model, data, q)
    tool_frame = data.oMf[tool_frame_id]
    notebook_viz.show_frame("tool0", tool_frame)
    update_object_pose(tool_sphere_handle, tool_frame * pin.SE3(np.eye(3), np.array([0.10, 0.0, 0.0])))
    time.sleep(dt)

q

## Add a custom frame to a robot model
Sometimes the frame you want does not exist yet. In that case you can add an operational frame and use it as a mounting point later.

In [ ]:
tool0_id = model.getFrameId("tool0")
tool0 = model.frames[tool0_id]

anchor_id = tool0.parentFrame
anchor = model.frames[anchor_id]

demo_mount = pin.Frame(
    "demo_mount",
    anchor.parentJoint,
    anchor_id,
    anchor.placement * pin.SE3(np.eye(3), np.array([0.0, 0.0, 0.3])),
    pin.FrameType.OP_FRAME,
)
demo_mount_id = model.addFrame(demo_mount, False)

data = model.createData()
pin.framesForwardKinematics(model, data, q_neutral)
notebook_viz.show_frame("demo_mount", data.oMf[demo_mount_id])

## Floating-base loading
The same robot description can be loaded with a different root joint. This is useful when the robot is not fixed to the world. The free-flyer contributes extra base coordinates, so here `nq` and `nv` are larger than in the fixed-base case.

In [ ]:
floating_robot = load_robot_description(
    "ur5_description",
    root_joint=pin.JointModelFreeFlyer(),
)

print(f"floating-base nq = {floating_robot.model.nq}, nv = {floating_robot.model.nv}")
notebook_viz.attach_robot(floating_robot, root_name="floating_ur5")
q_floating = pin.neutral(floating_robot.model)
q_floating[:3] = np.array([0.0, 0.0, 0.4])
notebook_viz.display(q_floating)
q_floating[:10]

## Move the floating base so the extra DOFs are visible
In the next cell we animate the base translation, the base orientation, and a little arm motion at the same time. That makes the floating-base degrees of freedom much easier to see.

In [ ]:
q = pin.neutral(floating_robot.model)
neutral_floating = pin.neutral(floating_robot.model)

for k in range(1000):
    t = 0.04 * k
    p = np.array([
        0.25 * np.sin(t),
        0.12 * np.cos(0.7 * t),
        0.35 + 0.08 * np.sin(1.3 * t),
    ])
    R = pin.utils.rotate('z', 0.8 * np.sin(0.5 * t)) @ pin.utils.rotate('y', 0.5 * np.sin(0.8 * t))
    q[:7] = pin.SE3ToXYZQUAT(pin.SE3(R, p))
    if floating_robot.model.nq > 7:
        q[7:] = neutral_floating[7:] + 0.35 * np.sin(t + np.linspace(0.0, 1.5, floating_robot.model.nq - 7))
    notebook_viz.display(q)
    time.sleep(0.01)

## Append two robot models
Here we attach an Allegro hand to the `tool0` frame of the UR5. The important idea is that `pin.appendModel(...)` creates one larger Pinocchio model.

In [ ]:
ur5 = load_robot_description("ur5_description")
allegro = load_robot_description("allegro_hand_description")

for frame in allegro.model.frames:
    frame.name = f"allegro_{frame.name}"

attach_frame_id = ur5.model.getFrameId("tool0")
combined_model, combined_visual_model = pin.appendModel(
    ur5.model,
    allegro.model,
    ur5.visual_model,
    allegro.visual_model,
    attach_frame_id,
    pin.SE3.Identity(),
)
_, combined_collision_model = pin.appendModel(
    ur5.model,
    allegro.model,
    ur5.collision_model,
    allegro.collision_model,
    attach_frame_id,
    pin.SE3.Identity(),
)

class CombinedRobot:
    def __init__(self, model, collision_model, visual_model):
        self.model = model
        self.collision_model = collision_model
        self.visual_model = visual_model

combined_robot = CombinedRobot(combined_model, combined_collision_model, combined_visual_model)
print(f"combined nq = {combined_model.nq}, nv = {combined_model.nv}")
notebook_viz.attach_robot(combined_robot, root_name="combined_robot")
notebook_viz.display(pin.neutral(combined_model))

## Animate the combined robot
We move the arm and the hand with different sinusoidal patterns so it is visually clear that they are now part of one larger Pinocchio model.

In [ ]:
q_combined = pin.neutral(combined_model)
neutral_combined = pin.neutral(combined_model)
arm_nq = ur5.model.nq
hand_nq = combined_model.nq - arm_nq

for k in range(1000):
    t = 0.04 * k
    q_combined[:] = neutral_combined
    q_combined[:arm_nq] += 0.45 * np.sin(t + np.linspace(0.0, 1.2, arm_nq))
    if hand_nq > 0:
        
        q_combined[arm_nq:] += 0.65 * np.sin(1.7 * t + np.linspace(0.0, 2.5, hand_nq))
    notebook_viz.display(q_combined)
    time.sleep(0.01)

## Exercise
Find a quadruped robot of your choice and a different robot arm on the `robot_descriptions` https://github.com/robot-descriptions/robot_descriptions.py repository. Attach the arm to the quadruped and animate the motion to verify correctness. You may need to add a custom frame to the quadruped first.

In [5]:
class CombinedRobot:
    def __init__(self, model, collision_model, visual_model):
        self.model = model
        self.collision_model = collision_model
        self.visual_model = visual_model

In [55]:
go2 = load_robot_description ("go2_description")
z1_mh = load_robot_description ("z1_mj_description")

for frame in go2.model.frames:
    frame.name = f"go2_{frame.name}"

go2_attach_frame_id = go2.model.getFrameId("go2_base") 

combined_model, combined_visual_model = pin.appendModel(
    go2.model,
    z1_mh.model,
    go2.visual_model,
    z1_mh.visual_model,
    go2_attach_frame_id,
    pin.SE3.Identity(),
)
_,combined_collision_model = pin.appendModel(
    go2.model,
    z1_mh.model,
    go2.collision_model,
    z1_mh.collision_model,
    go2_attach_frame_id,
    pin.SE3.Identity(),
)

comb_robot = CombinedRobot (combined_model, combined_collision_model, combined_visual_model)


In [56]:
print([frame.name for frame in go2.model.frames])

['go2_universe', 'go2_base', 'go2_FL_hip_joint', 'go2_FL_hip', 'go2_FL_thigh_joint', 'go2_FL_thigh', 'go2_FL_calf_joint', 'go2_FL_calf', 'go2_FL_calflower_joint', 'go2_FL_calflower', 'go2_FL_calflower1_joint', 'go2_FL_calflower1', 'go2_FL_foot_joint', 'go2_FL_foot', 'go2_FL_calf_rotor_joint', 'go2_FL_calf_rotor', 'go2_FL_thigh_rotor_joint', 'go2_FL_thigh_rotor', 'go2_FL_hip_rotor_joint', 'go2_FL_hip_rotor', 'go2_FR_hip_joint', 'go2_FR_hip', 'go2_FR_thigh_joint', 'go2_FR_thigh', 'go2_FR_calf_joint', 'go2_FR_calf', 'go2_FR_calflower_joint', 'go2_FR_calflower', 'go2_FR_calflower1_joint', 'go2_FR_calflower1', 'go2_FR_foot_joint', 'go2_FR_foot', 'go2_FR_calf_rotor_joint', 'go2_FR_calf_rotor', 'go2_FR_thigh_rotor_joint', 'go2_FR_thigh_rotor', 'go2_FR_hip_rotor_joint', 'go2_FR_hip_rotor', 'go2_Head_upper_joint', 'go2_Head_upper', 'go2_Head_lower_joint', 'go2_Head_lower', 'go2_RL_hip_joint', 'go2_RL_hip', 'go2_RL_thigh_joint', 'go2_RL_thigh', 'go2_RL_calf_joint', 'go2_RL_calf', 'go2_RL_calflow

In [57]:
data = comb_robot.model.createData()
q_neutral = pin.neutral (comb_robot.model)
pin.framesForwardKinematics(comb_robot.model, data, q_neutral)
base_frame_id = comb_robot.model.getFrameId("go2_base")


notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("base_link", data.oMf[base_frame_id])

FrameHandle(show_axes=True, axes_length=0.16, axes_radius=0.008, origin_radius=0.016, origin_color=(236, 236, 0), scale=1.0)

In [58]:
notebook_viz.attach_robot(comb_robot, root_name="combined_robot")
notebook_viz.display(pin.neutral(comb_robot.model))

In [60]:
# Animation
q_combined = pin.neutral (comb_robot.model)
neutral_combined = pin.neutral(combined_model)
nq_go2 = go2.model.nq
hand_nq = combined_model.nq - nq_go2 

# for k in range(1000):
#     t = 0.04 * k
#     q_combined[:] = neutral_combined
#     q_combined[:nq_go2] += 0.3 * np.sin(t + np.linspace(0.0, 0.9, nq_go2)) 
    
#     # q_combined [] += 0.65 * np.sin(1.7 * t + np.linspace(0.0, 2.5, nq_go2))
#     # if nq_go2 > 0:
#     #     q_combined[nq_go2-:] += 0.65 * np.sin(1.7 * t + np.linspace(0.0, 2.5, nq_go2))
#     notebook_viz.display(q_combined)
#     time.sleep(0.01)
print (f"Total DoF {comb_robot.model.nq}")
print (f"Go2 DoF {go2.model.nq}")
print (f"Arm DoF {z1_mh.nq}]")
print()

# for joint_idx in range (13):
#     comb_robot.model.names[joint_idx] = f"go2_{comb_robot.model.names[joint_idx]}"

# for joint_idx in range (6):
#     comb_robot.model.names[joint_idx+13] = f"z1_mh_{comb_robot.model.names[joint_idx]}"
for i, joint in enumerate (comb_robot.model.joints):
    if i <=12:
        comb_robot.model.names[i] = f"go2_{comb_robot.model.names[i]}"
    else:
        comb_robot.model.names[i] = f"z1_{comb_robot.model.names[i]}"
for i, joint in enumerate (comb_robot.model.joints):
    print (f"Joint {i:3d} | idx_q: {joint.idx_q:3d} | nq: {joint.nq} | name {comb_robot.model.names[i]} ")

Total DoF 18
Go2 DoF 12
Arm DoF 6]

Joint   0 | idx_q:  -1 | nq: 1 | name go2_universe 
Joint   1 | idx_q:   0 | nq: 1 | name go2_FL_hip_joint 
Joint   2 | idx_q:   1 | nq: 1 | name go2_FL_thigh_joint 
Joint   3 | idx_q:   2 | nq: 1 | name go2_FL_calf_joint 
Joint   4 | idx_q:   3 | nq: 1 | name go2_FR_hip_joint 
Joint   5 | idx_q:   4 | nq: 1 | name go2_FR_thigh_joint 
Joint   6 | idx_q:   5 | nq: 1 | name go2_FR_calf_joint 
Joint   7 | idx_q:   6 | nq: 1 | name go2_RL_hip_joint 
Joint   8 | idx_q:   7 | nq: 1 | name go2_RL_thigh_joint 
Joint   9 | idx_q:   8 | nq: 1 | name go2_RL_calf_joint 
Joint  10 | idx_q:   9 | nq: 1 | name go2_RR_hip_joint 
Joint  11 | idx_q:  10 | nq: 1 | name go2_RR_thigh_joint 
Joint  12 | idx_q:  11 | nq: 1 | name go2_RR_calf_joint 
Joint  13 | idx_q:  12 | nq: 1 | name z1_joint1 
Joint  14 | idx_q:  13 | nq: 1 | name z1_joint2 
Joint  15 | idx_q:  14 | nq: 1 | name z1_joint3 
Joint  16 | idx_q:  15 | nq: 1 | name z1_joint4 
Joint  17 | idx_q:  16 | nq: 1 |

In [61]:
go2_slice = slice (0, nq_go2)
arm_slice = slice (nq_go2, nq_go2 + z1_mh.nq)

In [73]:
for k in range(1000):
    t = 0.04 * k
    q_combined[:] = neutral_combined
    q_combined[go2_slice] += 0.3 * np.sin(t + np.linspace(0, 0.4, nq_go2))
    q_combined[arm_slice] += 0.2 * np.sin(t + np.linspace(0, 3, hand_nq)) + 2 
    notebook_viz.display(q_combined)
    time.sleep(0.01)